In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Controls — immediately after Drive mount.
DRIVE_ROOT='/content/drive/MyDrive/OpenPlaque'
OUTPUT_ROOT=DRIVE_ROOT + '/Joint_Three_Vessel_Template_Classifier_v1'
BRANCH='joint-three-vessel-template-classifier-from-main'


# OpenPlaque — joint RCA/LAD/LCX curved-template classifier

RCA and the accepted LAD calibrate the matching method **before** LCX candidates are ranked. The historical LCX curved series remains an unlabeled coronary template until source-space topology independently supports LCX identity. No stale historical LCX source centerline is used.


In [ ]:
import shutil, subprocess, sys
repo='/content/OpenPlaque'
shutil.rmtree(repo, ignore_errors=True)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/pazzani/OpenPlaque.git',repo], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',repo,'pytest','scikit-image'], check=True)
print('COMMIT:')
subprocess.run(['git','-C',repo,'rev-parse','HEAD'], check=True)
print('SEPARATE-PROCESS IMPORT CHECK:')
subprocess.run([sys.executable,'-c','import openplaque.joint_three_vessel_template_classifier as m; print(m.ALGORITHM)'], check=True)


In [ ]:
# Preflight all persistent scientific inputs before analysis.
from pathlib import Path
root=Path(DRIVE_ROOT)
required=[
 root/'Cache/Master_Coronary_Anatomy_Baseline_v2/master_anatomy_summary.json',
 root/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.npy',
 root/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.json',
 root/'Cache/LAD_Frozen_Proximal_Reacquisition_v1/combined_lad_centerline.csv',
 root/'PCAT_RCA_10_50/rca_centerline_smoothed_zyx.csv',
 root/'TotalSegmentator_Cardiovascular_Cache_v1/coronary_arteries/coronary_arteries.nii.gz',
 root/'TotalSegmentator_Cardiovascular_Cache_v1/coronary_arteries_LEGACY/coronary_arteries.nii.gz',
 root/'TotalSegmentator_Cardiovascular_Cache_v1/total/aorta.nii.gz',
 root/'UCLA_Plaque_Context_Verification/RCA_input/RCA_0000.nii.gz',
 root/'UCLA_Plaque_Context_Verification/LAD_input/LAD_0000.nii.gz',
 root/'UCLA_Plaque_Context_Verification/LCX_input/LCX_0000.nii.gz',
 root/'UCLA_Plaque_Context_Verification/nnunet_masks/RCA.nii.gz',
 root/'UCLA_Plaque_Context_Verification/nnunet_masks/LAD.nii.gz',
 root/'UCLA_Plaque_Context_Verification/nnunet_masks/LCX.nii.gz',
]
missing=[str(p) for p in required if not p.exists()]
print(f'INPUT PREFLIGHT: {len(required)-len(missing)}/{len(required)} present')
for p in required: print(('OK   ' if p.exists() else 'MISS '), p)
if missing: raise FileNotFoundError('Missing required cached inputs:\n'+'\n'.join(missing))


In [ ]:
subprocess.run([sys.executable,'-m','py_compile',repo+'/src/openplaque/joint_three_vessel_template_classifier.py'], check=True)
subprocess.run([sys.executable,'-m','pytest','-q',repo+'/tests/test_joint_three_vessel_template_classifier.py'], check=True)


In [ ]:
# Run in a fresh Python subprocess; print full stdout/stderr on failure.
import textwrap
runner=textwrap.dedent(f'''
from openplaque.joint_three_vessel_template_classifier import synthetic_joint_classifier_self_test, run
print('SELF TEST:', synthetic_joint_classifier_self_test(), flush=True)
result=run({DRIVE_ROOT!r}, {OUTPUT_ROOT!r})
print('STATUS:', result['summary']['status'], flush=True)
print('CALIBRATION:', result['summary']['calibration_passed'], result['summary']['selected_recipe'], flush=True)
print('REPORT:', result['report'], flush=True)
print('ZIP:', result['zip'], flush=True)
print('Drive search: https://drive.google.com/drive/u/0/search?q=OPENPLAQUE_JOINT_THREE_VESSEL_TEMPLATE_CLASSIFIER_REPORT_BACK.zip', flush=True)
''')
proc=subprocess.run([sys.executable,'-c',runner], text=True, capture_output=True)
print(proc.stdout)
if proc.stderr:
    print('--- WORKFLOW STDERR ---')
    print(proc.stderr)
if proc.returncode != 0:
    raise RuntimeError(f'Joint three-vessel workflow failed with exit code {proc.returncode}; traceback printed above')


In [ ]:
# Display final summary and QC figures.
import json
from IPython.display import display, Image
summary=json.loads((Path(OUTPUT_ROOT)/'summary.json').read_text())
display(summary)
for name in ['01_calibration_recipe_margins.png','02_candidate_three_template_scores.png','03_top_candidate_source_orthogonal_qc.png']:
    p=Path(OUTPUT_ROOT)/name
    if p.exists(): display(Image(filename=str(p)))
